In [1]:
from tensorflow import keras
from tensorflow.keras import layers
import pydot

I0000 00:00:1774556095.521441   24100 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774556095.667640   24100 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774556098.600804   24100 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
inputs   = keras.Input(shape=(3,), name="my_input")

features = layers.Dense(64, activation="relu")(inputs)

outputs  = layers.Dense(10, activation="softmax")(features)

model    = keras.Model(inputs=inputs, outputs=outputs)

E0000 00:00:1774556101.922468   24100 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_input (InputLayer)           │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 906 (3.54 KB)

 Trainable params: 906 (3.54 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# MULTI-INPUT, MULTI-OUTPUT MODELS

vocabulary_size = 10000
num_tags = 100
num_departments = 4

title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

features = layers.Concatenate()([title, text_body, tags])
features = layers.Dense(64, activation="relu")(features)
priority = layers.Dense(1, activation="sigmoid", name="priority")(features)

department = layers.Dense(num_departments, activation="softmax", name="department")(features)
model = keras.Model(inputs=[title, text_body, tags],
outputs=[priority, department])

In [5]:
import numpy as np

num_samples = 1280

title_data =     np.random.randint(0, 2, size=(num_samples, vocabulary_size))
text_body_data = np.random.randint(0, 2, size=(num_samples, vocabulary_size))
tags_data =      np.random.randint(0, 2, size=(num_samples, num_tags))
priority_data =  np.random.random(size=(num_samples, 1))
department_data =np.random.randint(0, 2, size=(num_samples, num_departments))

model.compile(optimizer="rmsprop",
loss=["mean_squared_error", "categorical_crossentropy"],
metrics=[["mean_absolute_error"], ["accuracy"]])
model.fit([title_data, text_body_data, tags_data],
[priority_data, department_data],
epochs=1)
model.evaluate([title_data, text_body_data, tags_data],
[priority_data, department_data])
priority_preds, department_preds = model.predict(
[title_data, text_body_data, tags_data])

# or

model.compile(optimizer="rmsprop",
loss={"priority": "mean_squared_error",
       "department":"categorical_crossentropy"},

metrics={"priority": ["mean_absolute_error"], 
         "department": ["accuracy"]})

model.fit({"title": title_data, "text_body": text_body_data,"tags": tags_data},
          {"priority": priority_data, "department": department_data},
epochs=1)
model.evaluate({"title": title_data, "text_body": text_body_data,
"tags": tags_data},
{"priority": priority_data, "department": department_data})
priority_preds, department_preds = model.predict(
{"title": title_data, "text_body": text_body_data, "tags": tags_data})

W0000 00:00:1774556103.047095   24100 cpu_allocator_impl.cc:82] Allocation of 102400000 exceeds 10% of free system memory.
W0000 00:00:1774556103.599156   24100 cpu_allocator_impl.cc:82] Allocation of 102400000 exceeds 10% of free system memory.


40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - department_accuracy: 0.2539 - department_loss: 21.9813 - loss: 22.2994 - priority_loss: 0.3180 - priority_mean_absolute_error: 0.4854


W0000 00:00:1774556106.597405   24100 cpu_allocator_impl.cc:82] Allocation of 102400000 exceeds 10% of free system memory.
W0000 00:00:1774556106.934408   24100 cpu_allocator_impl.cc:82] Allocation of 102400000 exceeds 10% of free system memory.


40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - department_accuracy: 0.5914 - department_loss: 13.8845 - loss: 14.2156 - priority_loss: 0.3311 - priority_mean_absolute_error: 0.4990


W0000 00:00:1774556108.223390   24100 cpu_allocator_impl.cc:82] Allocation of 102400000 exceeds 10% of free system memory.


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - department_accuracy: 0.2680 - department_loss: 25.0305 - loss: 25.3617 - priority_loss: 0.3311 - priority_mean_absolute_error: 0.4990
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - department_accuracy: 0.4977 - department_loss: 30.0424 - loss: 30.3736 - priority_loss: 0.3311 - priority_mean_absolute_error: 0.4990
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [6]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 10000)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_body           │ (None, 10000)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tags (InputLayer)   │ (None, 100)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 20100)     │          0 │ title[0][0],      │
│ (Concatenate)       │                   │            │ text_body[0][0],  │
│                     │                   │            │ tags[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │  1,286,464 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ priority (Dense)    │ (None, 1)         │         65 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department (Dense)  │ (None, 4)         │        260 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,573,580 (9.82 MB)

 Trainable params: 1,286,789 (4.91 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,286,791 (4.91 MB)

In [10]:
import graphviz

In [11]:
keras.utils.plot_model(model, "ticket_classifier.png")

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [12]:
model.layers

[<InputLayer name=title, built=True>,
 <InputLayer name=text_body, built=True>,
 <InputLayer name=tags, built=True>,
 <Concatenate name=concatenate, built=True>,
 <Dense name=dense_2, built=True>,
 <Dense name=priority, built=True>,
 <Dense name=department, built=True>]